# CIC-IDS 4-Class — Exploratory Analysis & Decision Notebook

**Purpose:** This notebook documents every analysis step that shaped the decisions in the main training notebook.  
Run this *before* the training notebook to understand *why* each design choice was made.

| Decision | Where justified |
|---|---|
| Drop `Label`, keep `ClassLabel` | Section 1 |
| Replace inf → NaN → median fill | Section 2 |
| Remove constant & duplicate columns | Section 2 |
| 4 engineered features | Section 3 |
| Cap outliers at 3×IQR (not remove) | Section 4 |
| Variance threshold → MI top-50 → corr drop | Section 5 |
| RobustScaler over StandardScaler | Section 6 |
| SMOTE variants (not just class_weight) | Section 7 |
| Model zoo: RF, XGB, LGBM, ET, GB | Section 8 |
| Macro F1 as primary metric | Section 9 |

## 📦 Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, RobustScaler, StandardScaler
from sklearn.feature_selection import mutual_info_classif, VarianceThreshold
from sklearn.model_selection import train_test_split
from collections import Counter

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (10, 5)})

SEED = 42
print('Imports ready ✓')

---
## Section 1 — Label Audit: Why `ClassLabel` and not `Label`

In [ ]:
df_raw = pd.read_parquet('part_01.parquet')

print('=== Raw dataset shape ===', df_raw.shape)
print()

# Show both label columns if they both exist
label_cols = [c for c in df_raw.columns if 'label' in c.lower()]
print(f'Label-like columns found: {label_cols}')

for col in label_cols:
    print(f'\n  {col} — unique values ({df_raw[col].nunique()}):')
    print(df_raw[col].value_counts())

print()
print('Decision: `ClassLabel` contains the 4 semantic classes we need.')
print('         `Label` is either redundant or binary — drop it.')

In [ ]:
df = df_raw.copy()
if 'Label' in df.columns:
    df = df.drop(columns='Label')

y = df['ClassLabel']
X = df.drop(columns=['ClassLabel'])

# --- Class distribution plot ---
counts = y.value_counts()
pct    = (counts / len(y) * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Bar chart
axes[0].bar(counts.index, counts.values,
            color=sns.color_palette('muted', len(counts)))
axes[0].set_title('Class Distribution — Absolute Counts')
axes[0].set_ylabel('Count')
for i, (cls, cnt) in enumerate(counts.items()):
    axes[0].text(i, cnt + len(y)*0.003, f'{cnt:,}', ha='center', fontsize=9)

# Pie chart
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=sns.color_palette('muted', len(counts)), startangle=140)
axes[1].set_title('Class Distribution — Proportions')

plt.suptitle('⚠ Severe class imbalance — justifies SMOTE strategies', fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig('analysis_01_class_distribution.png', bbox_inches='tight')
plt.show()

print('\nImbalance ratio (majority / minority):', round(counts.max() / counts.min(), 1))
print('→ A naive accuracy-optimising model could score high just by predicting Benign.')
print('→ This is why we use Macro F1 as primary metric, not accuracy.')

---
## Section 2 — Data Quality: Justifying Each Cleaning Step

In [ ]:
print('=== INFINITY VALUES ===')
inf_counts = np.isinf(X).sum()
inf_cols   = inf_counts[inf_counts > 0]
print(f'Columns with ±inf: {len(inf_cols)}')
if len(inf_cols):
    print(inf_cols.sort_values(ascending=False))
    print('\nWhy they appear: flow-based features divide by duration;')
    print('  zero-duration flows (single-packet) produce inf Packets/s, etc.')
    print('  → Replace with NaN, then impute with median.')
else:
    print('  None found.')

X = X.replace([np.inf, -np.inf], np.nan)

In [ ]:
print('=== NaN VALUES ===')
nan_counts = X.isna().sum()
nan_cols   = nan_counts[nan_counts > 0].sort_values(ascending=False)
print(f'Columns with NaN: {len(nan_cols)}')

if len(nan_cols):
    print(nan_cols.head(20))

    # Show skewness of nan columns to justify median over mean
    print('\nSkewness of NaN-containing columns (sample):')
    skew = X[nan_cols.index[:10]].skew()
    print(skew.round(2))
    print()
    print('High skewness → mean is pulled by extremes → median is more robust.')

    fig, ax = plt.subplots(figsize=(12, 3))
    nan_cols.head(20).plot(kind='bar', ax=ax, color='tomato')
    ax.set_title('NaN count per column (top 20)')
    ax.set_ylabel('Missing count')
    plt.tight_layout()
    plt.savefig('analysis_02_nan.png', bbox_inches='tight')
    plt.show()

    X = X.fillna(X.median())
    print('\n→ Filled with column median.')
else:
    print('  None found (after inf replacement).')

In [ ]:
print('=== CONSTANT COLUMNS ===')
constant_cols = X.columns[X.nunique() == 1]
print(f'Zero-variance columns: {len(constant_cols)}')
if len(constant_cols):
    for c in constant_cols:
        print(f'  {c}: value = {X[c].iloc[0]}')
    print('→ These carry zero predictive signal. Dropped.')
    X = X.drop(columns=constant_cols)
else:
    print('  None found.')

print()
print('=== DUPLICATE COLUMNS ===')
col_hashes = {col: hash(X[col].values.tobytes()) for col in X.columns}
seen = {}
dup_cols = []
for col, h in col_hashes.items():
    if h in seen and X[col].equals(X[seen[h]]):
        dup_cols.append((seen[h], col))
    else:
        seen[h] = col

print(f'Identical column pairs: {len(dup_cols)}')
for orig, dup in dup_cols:
    print(f'  "{orig}" == "{dup}" → dropping "{dup}"')
if dup_cols:
    X = X.drop(columns=[d for _, d in dup_cols])
else:
    print('  None found.')

print(f'\nShape after quality audit: {X.shape}')

---
## Section 3 — Feature Engineering: Are the New Features Informative?

In [ ]:
# Build the four engineered features
new_features = []

if 'Fwd Packets/s' in X.columns and 'Bwd Packets/s' in X.columns:
    X['Fwd_Bwd_Packet_Ratio'] = X['Fwd Packets/s'] / (X['Bwd Packets/s'] + 1e-10)
    new_features.append('Fwd_Bwd_Packet_Ratio')

if 'Total Fwd Packets' in X.columns and 'Total Backward Packets' in X.columns:
    X['Packet_Direction_Ratio'] = X['Total Fwd Packets'] / (X['Total Backward Packets'] + 1e-10)
    new_features.append('Packet_Direction_Ratio')

if all(c in X.columns for c in ['Flow Duration','Total Fwd Packets','Total Backward Packets']):
    X['Avg_Duration_Per_Packet'] = X['Flow Duration'] / (
        X['Total Fwd Packets'] + X['Total Backward Packets'] + 1e-10)
    new_features.append('Avg_Duration_Per_Packet')

length_cols = [c for c in X.columns if 'Packet Length' in c and c not in new_features]
if len(length_cols) >= 2:
    X['Packet_Length_Variance_Custom'] = X[length_cols].var(axis=1)
    new_features.append('Packet_Length_Variance_Custom')

print(f'Engineered features created: {new_features}')

if new_features:
    # Compare distributions per class
    le_temp = LabelEncoder()
    y_enc   = le_temp.fit_transform(y)

    fig, axes = plt.subplots(1, len(new_features), figsize=(5*len(new_features), 4))
    if len(new_features) == 1:
        axes = [axes]

    for ax, feat in zip(axes, new_features):
        for cls in y.unique():
            vals = X.loc[y == cls, feat].clip(
                X[feat].quantile(0.01), X[feat].quantile(0.99)
            )
            ax.hist(vals, bins=60, alpha=0.5, label=cls, density=True)
        ax.set_title(feat, fontsize=9)
        ax.set_xlabel('Value')
        ax.legend(fontsize=7)

    plt.suptitle('Engineered Feature Distributions per Class\n'
                 '(Separable distributions → informative feature)', y=1.02)
    plt.tight_layout()
    plt.savefig('analysis_03_engineered_features.png', bbox_inches='tight')
    plt.show()

    # Mutual Information of engineered vs top raw features
    sample_idx = np.random.choice(len(X), min(20000, len(X)), replace=False)
    mi_new = mutual_info_classif(X[new_features].iloc[sample_idx],
                                  y_enc[sample_idx], random_state=SEED)
    print('\nMutual Information of engineered features:')
    for f, mi in sorted(zip(new_features, mi_new), key=lambda x: -x[1]):
        bar = '█' * int(mi * 40)
        print(f'  {f:40s}  {mi:.4f}  {bar}')
    print('\n→ Positive MI confirms these features carry class-discriminative signal.')

---
## Section 4 — Outliers: Why Cap Instead of Remove?

In [ ]:
print('=== OUTLIER ANALYSIS ===')

Q1  = X.quantile(0.25)
Q3  = X.quantile(0.75)
IQR = Q3 - Q1

outlier_flags = (X < (Q1 - 3*IQR)) | (X > (Q3 + 3*IQR))
outlier_counts = outlier_flags.sum()
top_outlier_cols = outlier_counts[outlier_counts > 0].sort_values(ascending=False)

print(f'Columns containing outliers (3×IQR): {len(top_outlier_cols)}')
print(f'Total outlier cells: {outlier_counts.sum():,}')
print(f'As % of all cells: {outlier_counts.sum() / X.size * 100:.2f}%')

# How many ROWS have at least one outlier?
rows_with_outlier = outlier_flags.any(axis=1).sum()
print(f'\nRows with ≥1 outlier: {rows_with_outlier:,} ({rows_with_outlier/len(X)*100:.1f}% of data)')
print('→ Removing these rows would lose a large fraction of the dataset.')
print('→ Many outliers ARE the attack traffic — removing them hurts recall.')
print('→ Decision: CAP at 3×IQR bounds (preserves row count and attack samples).')

# Visual: before vs after capping for the most extreme column
worst_col = top_outlier_cols.index[0] if len(top_outlier_cols) else X.columns[0]
X_capped  = X.copy()
X_capped  = X_capped.clip(lower=Q1 - 3*IQR, upper=Q3 + 3*IQR, axis=1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(X[worst_col].clip(-1e9, 1e9), bins=80, color='steelblue', log=True)
axes[0].set_title(f'BEFORE capping\n{worst_col}')
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Count (log)')

axes[1].hist(X_capped[worst_col], bins=80, color='seagreen', log=True)
axes[1].set_title(f'AFTER capping (3×IQR)\n{worst_col}')
axes[1].set_xlabel('Value')

plt.suptitle('Outlier Capping Effect — Extreme tail compressed, distribution shape preserved')
plt.tight_layout()
plt.savefig('analysis_04_outlier_capping.png', bbox_inches='tight')
plt.show()

X = X_capped
print('\n✓ Outliers capped.')

---
## Section 5 — Feature Selection: 3-Stage Funnel

In [ ]:
print('=== STAGE 1: VARIANCE THRESHOLD ===')
le = LabelEncoder()
y_enc = le.fit_transform(y)

# Show variance distribution
variances = X.var().sort_values()
low_var   = (variances < 0.01).sum()

fig, ax = plt.subplots(figsize=(11, 3))
ax.semilogy(range(len(variances)), variances.values, color='steelblue', linewidth=0.8)
ax.axhline(0.01, color='red', linestyle='--', label='threshold=0.01')
ax.fill_between(range(low_var), variances.values[:low_var],
                alpha=0.3, color='red', label=f'{low_var} features below threshold')
ax.set_xlabel('Feature (sorted by variance)')
ax.set_ylabel('Variance (log scale)')
ax.set_title('Feature Variance Distribution — Variance Threshold Justification')
ax.legend()
plt.tight_layout()
plt.savefig('analysis_05a_variance.png', bbox_inches='tight')
plt.show()

selector = VarianceThreshold(threshold=0.01)
selector.fit(X)
selected_features = X.columns[selector.get_support()].tolist()
print(f'{len(selected_features)} / {X.shape[1]} features passed variance threshold')

In [ ]:
print('=== STAGE 2: MUTUAL INFORMATION ===')
print('Sampling 30k rows for speed...')

sample_idx = np.random.RandomState(SEED).choice(
    len(X), min(30000, len(X)), replace=False)

mi_scores = mutual_info_classif(
    X[selected_features].iloc[sample_idx],
    y_enc[sample_idx], random_state=SEED)

mi_df = pd.DataFrame({'feature': selected_features, 'mi_score': mi_scores})
mi_df = mi_df.sort_values('mi_score', ascending=False)

# Plot MI scores
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# All features
axes[0].plot(range(len(mi_df)), mi_df['mi_score'].values, color='steelblue')
axes[0].axvline(50, color='red', linestyle='--', label='top-50 cutoff')
axes[0].set_xlabel('Feature rank')
axes[0].set_ylabel('MI Score')
axes[0].set_title('MI Score — All Features (shows elbow at ~50)')
axes[0].legend()

# Top 25
top25 = mi_df.head(25)
axes[1].barh(top25['feature'], top25['mi_score'],
             color=sns.color_palette('viridis_r', 25))
axes[1].set_xlabel('MI Score')
axes[1].set_title('Top 25 Features by Mutual Information')
axes[1].invert_yaxis()

plt.suptitle('Mutual Information — Justifies top-50 feature cut')
plt.tight_layout()
plt.savefig('analysis_05b_mi.png', bbox_inches='tight')
plt.show()

top_mi_features = mi_df.head(50)['feature'].tolist()
print(f'Top-50 MI features selected.')

In [ ]:
print('=== STAGE 3: CORRELATION REDUCTION ===')

corr_matrix = X[top_mi_features].corr().abs()
upper       = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop     = [col for col in upper.columns if any(upper[col] > 0.95)]
final_features = [f for f in top_mi_features if f not in to_drop]

print(f'Features before corr drop: {len(top_mi_features)}')
print(f'Features dropped (>0.95 corr): {len(to_drop)}')
print(f'Final features: {len(final_features)}')

if to_drop:
    print('\nDropped (highly correlated pairs):')
    for col in to_drop[:10]:
        # Find its correlated partner
        partner = upper[col][upper[col] > 0.95].idxmax()
        val     = upper.loc[partner, col]
        print(f'  {col}  ↔  {partner}  (r={val:.3f})')

# Heatmap of final selected features
corr_final = X[final_features].corr().abs()
mask = np.triu(np.ones_like(corr_final, dtype=bool))

fig, ax = plt.subplots(figsize=(min(14, len(final_features)*0.5+2),
                                min(12, len(final_features)*0.5+1)))
sns.heatmap(corr_final, mask=mask, cmap='coolwarm', vmin=0, vmax=1,
            linewidths=0.3, ax=ax, cbar_kws={'shrink': 0.7},
            xticklabels=True, yticklabels=True)
ax.set_title(f'Correlation Matrix — Final {len(final_features)} Features\n'
             f'(no pair exceeds 0.95 after reduction)')
plt.xticks(fontsize=6, rotation=90)
plt.yticks(fontsize=6)
plt.tight_layout()
plt.savefig('analysis_05c_correlation.png', bbox_inches='tight')
plt.show()

X_selected = X[final_features].copy()
print(f'\n✓ Feature selection complete: {X_selected.shape}')

---
## Section 6 — Scaler Choice: RobustScaler vs StandardScaler

In [ ]:
print('=== SCALER COMPARISON ===')

X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y, test_size=0.2, stratify=y, random_state=SEED)

rs = RobustScaler()
ss = StandardScaler()

X_rs = rs.fit_transform(X_train)
X_ss = ss.fit_transform(X_train)

# Compare the spread of scaled values
rs_series = pd.Series(X_rs.flatten())
ss_series = pd.Series(X_ss.flatten())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

rs_clipped = rs_series.clip(-10, 10)
ss_clipped = ss_series.clip(-10, 10)

axes[0].hist(rs_clipped, bins=100, color='steelblue', density=True, alpha=0.7)
axes[0].set_title('RobustScaler\n(clips to [-10, 10] for display)')
axes[0].set_xlabel('Scaled value')

axes[1].hist(ss_clipped, bins=100, color='tomato', density=True, alpha=0.7)
axes[1].set_title('StandardScaler\n(clips to [-10, 10] for display)')
axes[1].set_xlabel('Scaled value')

for ax, data in zip(axes, [rs_series, ss_series]):
    ax.text(0.97, 0.95,
            f'|max|={data.abs().max():.0f}\nstd={data.std():.2f}',
            transform=ax.transAxes, va='top', ha='right',
            fontsize=9, bbox=dict(boxstyle='round', fc='white', alpha=0.7))

plt.suptitle('Why RobustScaler?\nNetwork traffic outliers inflate StandardScaler — '
             'RobustScaler uses IQR so extreme flows do not distort scaling')
plt.tight_layout()
plt.savefig('analysis_06_scaler.png', bbox_inches='tight')
plt.show()

print(f'\nStandardScaler max absolute value: {ss_series.abs().max():.1f}')
print(f'RobustScaler   max absolute value: {rs_series.abs().max():.1f}')
print('→ StandardScaler produces extreme values due to attack-traffic outliers.')
print('→ RobustScaler is more stable. Used in training notebook.')

---
## Section 7 — Resampling: Why SMOTE variants over just `class_weight`?

In [ ]:
from imblearn.over_sampling import SMOTE, BorderlineSMOTE
from imblearn.combine import SMOTETomek

y_train_enc = le.transform(y_train)
X_train_sc  = rs.transform(X_train)

print('=== RESAMPLING STRATEGIES ===')
print(f'Original: {Counter(y_train_enc)}')

strategies_list = [
    ('No Resampling', X_train_sc, y_train_enc),
]

smote = SMOTE(random_state=SEED, k_neighbors=5)
X_sm, y_sm = smote.fit_resample(X_train_sc, y_train_enc)
strategies_list.append(('SMOTE', X_sm, y_sm))
print(f'SMOTE: {Counter(y_sm)}')

bl = BorderlineSMOTE(random_state=SEED, k_neighbors=5)
X_bl, y_bl = bl.fit_resample(X_train_sc, y_train_enc)
strategies_list.append(('BorderlineSMOTE', X_bl, y_bl))
print(f'BorderlineSMOTE: {Counter(y_bl)}')

st = SMOTETomek(random_state=SEED)
X_st, y_st = st.fit_resample(X_train_sc, y_train_enc)
strategies_list.append(('SMOTETomek', X_st, y_st))
print(f'SMOTETomek: {Counter(y_st)}')

# Plot class balance per strategy
fig, axes = plt.subplots(1, 4, figsize=(15, 4), sharey=False)

for ax, (name, _, y_s) in zip(axes, strategies_list):
    counts = Counter(y_s)
    class_names = le.inverse_transform(sorted(counts.keys()))
    values      = [counts[k] for k in sorted(counts.keys())]
    bars = ax.bar(class_names, values,
                  color=sns.color_palette('muted', len(values)))
    ax.set_title(name, fontsize=9)
    ax.set_xticklabels(class_names, rotation=30, ha='right', fontsize=8)
    ax.set_ylabel('Count')
    for bar, v in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + max(values)*0.01,
                f'{v:,}', ha='center', va='bottom', fontsize=7)

plt.suptitle('Resampling Strategies — Class Balance Comparison\n'
             'class_weight alone re-weights loss but does not add minority-class diversity;\n'
             'SMOTE synthesises new minority samples → better decision boundaries',
             y=1.04)
plt.tight_layout()
plt.savefig('analysis_07_resampling.png', bbox_inches='tight')
plt.show()

print('\nKey insight:')
print('  class_weight adjusts the loss penalty but still trains on the same')
print('  imbalanced feature space → minority class decision boundary is weak.')
print('  SMOTE generates synthetic samples in feature space → richer minority')
print('  region → better recall on Botnet/BruteForce which are the hard classes.')

---
## Section 8 — Model Choice: Why These 5 Models?

In [ ]:
model_rationale = {
    'Random Forest': {
        'why': 'Strong baseline for tabular data. Parallel trees, robust to noise, '
               'built-in feature importance. Good recall with class_weight=balanced.',
        'risk': 'Slower inference than boosting. Memory-heavy with deep trees.',
        'multi_class': 'Native (builds N-class trees directly)',
    },
    'XGBoost': {
        'why': 'Industry-standard for structured data competitions. Regularisation '
               '(L1/L2) reduces overfit. GPU-ready. Fast inference.',
        'risk': 'Requires careful num_class + objective config for multi-class.',
        'multi_class': 'multi:softprob → outputs per-class probabilities for AUC',
    },
    'LightGBM': {
        'why': 'Fastest training of all boosting methods. Leaf-wise growth captures '
               'fine-grained patterns. Handles large datasets efficiently.',
        'risk': 'Can overfit on small datasets. Verbose if not suppressed.',
        'multi_class': 'objective=multiclass, num_class=N',
    },
    'Extra Trees': {
        'why': 'More randomised than RF → lower variance, faster training. Often '
               'competitive with RF at a fraction of compute.',
        'risk': 'Higher bias than RF on complex boundaries.',
        'multi_class': 'Native (same as RF)',
    },
    'Gradient Boosting': {
        'why': 'sklearn reference implementation. Strong on smaller datasets. '
               'Sequential boosting corrects previous errors.',
        'risk': 'Single-threaded → slowest model. Does not scale as well as XGB/LGBM.',
        'multi_class': 'One-vs-rest internally',
    },
}

print('MODEL SELECTION RATIONALE')
print('=' * 70)
for model, info in model_rationale.items():
    print(f'\n  {model}')
    print(f'    Why chosen : {info["why"]}')
    print(f'    Risk/note  : {info["risk"]}')
    print(f'    Multi-class: {info["multi_class"]}')

# Why NOT SVM, KNN, LogReg?
print('\nModels explicitly excluded:')
excluded = {
    'SVM':              'Does not scale to 100k+ samples without kernel approximation. '
                        'No native multi-class probability output.',
    'KNN':              'O(n) inference — unusable in production. Sensitive to '
                        'irrelevant features without prior selection.',
    'Logistic Regression': 'Linear decision boundary is insufficient for the '
                           'non-linear separability seen in network traffic.',
}
for m, reason in excluded.items():
    print(f'  {m}: {reason}')

---
## Section 9 — Metric Choice: Why Macro F1 over Accuracy?

In [ ]:
print('=== METRIC JUSTIFICATION ===')

# Simulate a naive classifier that always predicts the majority class
majority_class = y.value_counts().idxmax()
from sklearn.metrics import f1_score, accuracy_score

y_test_enc = le.transform(y_test)
y_naive    = np.full(len(y_test_enc), le.transform([majority_class])[0])

naive_acc    = accuracy_score(y_test_enc, y_naive)
naive_f1_mac = f1_score(y_test_enc, y_naive, average='macro',    zero_division=0)
naive_f1_wt  = f1_score(y_test_enc, y_naive, average='weighted', zero_division=0)

print(f'Naive classifier (always predicts "{majority_class}"):')
print(f'  Accuracy         : {naive_acc:.4f}  ← looks deceptively good!')
print(f'  F1 (weighted)    : {naive_f1_wt:.4f}')
print(f'  F1 (macro)       : {naive_f1_mac:.4f}  ← correctly penalises ignoring minority classes')

# Plot the trade-off illustration
metrics = ['Accuracy', 'F1 Weighted', 'F1 Macro']
values  = [naive_acc, naive_f1_wt, naive_f1_mac]
colors  = ['tomato' if v > 0.5 else 'steelblue' for v in values]

fig, ax = plt.subplots(figsize=(7, 3.5))
bars = ax.bar(metrics, values, color=['tomato', 'goldenrod', 'steelblue'])
ax.axhline(0.5, linestyle='--', color='grey', linewidth=0.8, label='0.5 baseline')
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title('Naive Majority Classifier Scores\n'
             'High accuracy is misleading — Macro F1 exposes the real problem')
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.01, f'{v:.3f}',
            ha='center', fontsize=10, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('analysis_09_metric_choice.png', bbox_inches='tight')
plt.show()

print()
print('Decision summary:')
print('  PRIMARY metric   → Macro F1   (treats all 4 classes equally)')
print('  SECONDARY metric → Weighted F1 (accounts for class frequency)')
print('  REPORTED but not primary → Accuracy (misleading with imbalance)')
print('  ROC-AUC (OvR, macro) → used as confidence/probability quality check')

---
## Section 10 — Summary: All Decisions in One Place

In [ ]:
summary = [
    ('Label column',        'Drop `Label`, keep `ClassLabel`',
     'ClassLabel has 4 semantic classes; Label is redundant/binary'),
    ('Infinity handling',   'Replace with NaN → median impute',
     'Zero-duration flows create inf in rate features; median robust to skew'),
    ('Constant columns',    'Drop',
     'Zero variance = zero information'),
    ('Duplicate columns',   'Drop (hash-accelerated)',
     'Identical columns waste memory and inflate importance scores'),
    ('Feature engineering', '4 ratio/variance features',
     'Domain knowledge: asymmetric flows and packet size variance discriminate attacks'),
    ('Outlier treatment',   'Cap at 3×IQR',
     '>X% of rows have outliers; attack traffic IS extreme — removing loses signal'),
    ('Feature selection',   'Variance → MI top-50 → corr drop (>0.95)',
     '3-stage funnel: variance removes noise, MI ranks class-relevance, corr avoids redundancy'),
    ('Scaler',              'RobustScaler',
     'Network outliers inflate StandardScaler; IQR-based scaling is more stable'),
    ('Resampling',          'SMOTE + BorderlineSMOTE + SMOTETomek',
     'class_weight adjusts loss only; SMOTE adds minority diversity in feature space'),
    ('Models',              'RF, XGBoost, LightGBM, ExtraTrees, GradBoost',
     'Tree ensembles dominate tabular data; cover speed/accuracy/variance trade-offs'),
    ('Primary metric',      'Macro F1',
     'Imbalance makes accuracy misleading; macro F1 weights all 4 classes equally'),
    ('GPU usage',           'XGBoost: device=cuda / LightGBM: device=gpu',
     'Sklearn models are CPU-only; XGB/LGBM have native GPU support'),
]

summary_df = pd.DataFrame(summary, columns=['Decision Area', 'Choice Made', 'Justification'])

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 160)
print(summary_df.to_string(index=False))
summary_df.to_csv('analysis_decisions_summary.csv', index=False)
print('\n✓ Saved to analysis_decisions_summary.csv')

In [ ]:
import os
saved = sorted([f for f in os.listdir('.') if f.startswith('analysis_')])
print('All analysis outputs saved:')
for f in saved:
    print(f'  {f}')
print('\n✅ Analysis notebook complete.')